# Designer Agent Notebook

This notebook showcases the Designer Agent, the primary interface for code generation.

## Purpose
The Designer Agent is responsible for translating user requirements into quantum circuits. This notebook demonstrates:

1.  **Agent Setup**: Initializing the Designer Agent with RAG capabilities.
2.  **Task Execution**: Sending natural language tasks (e.g., "Create a Teleportation circuit") to the agent.
3.  **Code Production**: Verifying that the agent produces syntactically correct Braket code based on the input.

## Usage
Use this notebook to interact with the Designer Agent and generate initial circuit implementations.


In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.braket_rag_code_assistant.config import get_config, setup_logging
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator
from src.agents.designer import DesignerAgent

# Setup logging
setup_logging()

2026-03-15 02:01:28 | INFO     | src.braket_rag_code_assistant.config.logging:setup_all:137 | Logging configuration completed


### Initialize RAG Components
We need to initialize the underlying RAG components first:
1. Load the KnowledgeBase and its entries
2. Build or load the vector index
3. Create Retriever and Generator

In [2]:
# Define paths
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
VECTOR_INDEX_PATH = project_root / "data" / "models" / "vector_index"

# Initialize Knowledge Base and load entries
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
num_loaded = kb.load_from_directory()
print(f"Loaded {num_loaded} entries from knowledge base")

# Try to load existing index, or build new one
try:
    kb.load_index(VECTOR_INDEX_PATH)
    print(f"Loaded vector index: {kb.vector_store.size()} entries")
except FileNotFoundError:
    print("No existing index found, building new one...")
    kb.index_entries()
    kb.save_index(VECTOR_INDEX_PATH)
    print(f"Built and saved vector index: {kb.vector_store.size()} entries")

# Initialize RAG components (uses Ollama by default)
retriever = Retriever(knowledge_base=kb, use_hybrid_scoring=True)

try:
    generator = Generator(retriever=retriever)
    print("\n✅ RAG components initialized.")
except Exception as e:
    print(f"\n⚠️ Error initializing Generator: {e}")
    print("Make sure Ollama is running locally.")
    generator = None

2026-03-15 02:01:28 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Hackathon\Amazons\config\config.json


2026-03-15 02:01:28,205 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-15 02:01:28 | INFO     | src.rag.embeddings:_init_aws:120 | Using AWS Nova Multimodal Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-15 02:01:28 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-15 02:01:28 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-15 02:01:28 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-15 02:01:28 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-15 02:01:28 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-15 02:01:28 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-15 02:01:28 | INFO     | src.rag.kno

Loaded 140 entries from knowledge base
Loaded vector index: 140 entries

✅ RAG components initialized.


### Initialize Designer Agent
The Designer Agent combines retrieval and generation to produce Braket code.

In [3]:
if generator is not None:
    # Initialize Designer Agent
    designer = DesignerAgent(retriever=retriever, generator=generator)
    print("✅ Designer Agent initialized.")
else:
    designer = None
    print("⚠️ Cannot initialize Designer Agent without Generator.")
    print("Make sure Ollama is running locally.")

2026-03-15 02:01:28 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent


✅ Designer Agent initialized.


### Generate Circuit
Let's ask the agent to design a circuit.

In [4]:
if designer is not None:
    task = {
        "query": "Create a circuit for Quantum Teleportation",
        "algorithm": "teleportation"
    }

    try:
        result = designer.execute(task)
        
        if result['success']:
            print("✅ Successfully generated circuit!")
            print("\nCode:")
            print("-" * 40)
            print(result['code'])
            print("-" * 40)
            print(f"\nAlgorithm: {result.get('algorithm', 'N/A')}")
            print(f"Context used: {result.get('context_used', 'N/A')} examples")
        else:
            print(f"❌ Failed to generate circuit: {result.get('error')}")
            if 'code' in result:
                print("\nGenerated code (with errors):")
                print(result['code'])
            
    except Exception as e:
        print(f"Error executing task: {e}")
else:
    print("Designer Agent not available.")

2026-03-15 02:01:28 | INFO     | src.rag.generator:generate:199 | Retrieving context for query: Create a circuit for Quantum Teleportation...
2026-03-15 02:01:30 | INFO     | src.rag.generator:generate:240 | Generating code using aws/amazon.nova-pro-v1:0
2026-03-15 02:01:33 | INFO     | src.rag.generator:generate:308 | ✅ Code generation completed


✅ Successfully generated circuit!

Code:
----------------------------------------
from braket.circuits import Circuit

# Initialize a quantum circuit with 3 qubits
circuit = Circuit()

# Prepare the initial state to be teleported
circuit.h(0)  # Apply Hadamard gate to qubit 0 to create a superposition

# Prepare the Bell state between qubits 1 and 2
circuit.h(1)  # Apply Hadamard gate to qubit 1
circuit.cnot(1, 2)  # Apply CNOT gate with qubit 1 as control and qubit 2 as target

# Entangle qubit 0 with qubit 1
circuit.cnot(0, 1)  # Apply CNOT gate with qubit 0 as control and qubit 1 as target
circuit.h(0)  # Apply Hadamard gate to qubit 0

# Measure qubits 0 and 1
circuit.measure([0, 1])  # Measure qubits 0 and 1

# Apply conditional gates based on the measurement results
circuit.cnot(1, 2)  # Apply CNOT gate with qubit 1 as control and qubit 2 as target
circuit.cz(0, 2)  # Apply CZ gate with qubit 0 as control and qubit 2 as target

# Measure qubit 2 to obtain the teleported state
cir

### Try More Examples
Let's try a few more circuit generation tasks.

In [5]:
if designer is not None:
    test_tasks = [
        {"query": "Create a Bell state circuit", "algorithm": "bell_state"},
        {"query": "Implement Grover's search for 2 qubits", "algorithm": "grover"},
        {"query": "Build a 3-qubit GHZ state", "algorithm": "ghz_state"},
    ]
    
    for i, task in enumerate(test_tasks, 1):
        print(f"\n{'='*60}")
        print(f"Task {i}: {task['query']}")
        print('='*60)
        
        try:
            result = designer.execute(task)
            if result['success']:
                print("✅ Success")
                # Show first 10 lines of code
                code_lines = result['code'].split('\n')[:10]
                print("Code preview:")
                for line in code_lines:
                    print(f"  {line}")
                if len(result['code'].split('\n')) > 10:
                    print("  ...")
            else:
                print(f"❌ Failed: {result.get('error')}")
        except Exception as e:
            print(f"Error: {e}")

2026-03-15 02:01:33 | INFO     | src.rag.generator:generate:199 | Retrieving context for query: Create a Bell state circuit...



Task 1: Create a Bell state circuit


2026-03-15 02:01:34 | INFO     | src.rag.generator:generate:240 | Generating code using aws/amazon.nova-pro-v1:0
2026-03-15 02:01:35 | INFO     | src.rag.generator:generate:308 | ✅ Code generation completed
2026-03-15 02:01:35 | INFO     | src.rag.generator:generate:199 | Retrieving context for query: Implement Grover's search for 2 qubits...


✅ Success
Code preview:
  from braket.circuits import Circuit
  
  # Initialize a quantum circuit with 2 qubits
  bell_circuit = Circuit()
  
  # Apply a Hadamard gate to the first qubit to create superposition
  bell_circuit.h(0)
  
  # Apply a CNOT gate with the first qubit as control and the second qubit as target
  # This creates entanglement between the two qubits
  ...

Task 2: Implement Grover's search for 2 qubits


2026-03-15 02:01:36 | INFO     | src.rag.generator:generate:240 | Generating code using aws/amazon.nova-pro-v1:0
2026-03-15 02:01:38 | INFO     | src.rag.generator:generate:308 | ✅ Code generation completed
2026-03-15 02:01:38 | INFO     | src.rag.generator:generate:199 | Retrieving context for query: Build a 3-qubit GHZ state...


✅ Success
Code preview:
  from braket.circuits import Circuit
  
  # Initialize a quantum circuit with 2 qubits
  circuit = Circuit()
  
  # Apply Hadamard gates to create a superposition of all states
  circuit.h(0).h(1)
  
  # Apply a phase shift to the target state |11>
  circuit.x(0).x(1).h(1).cx(0, 1).h(1).x(0).x(1)
  ...

Task 3: Build a 3-qubit GHZ state


2026-03-15 02:01:39 | INFO     | src.rag.generator:generate:240 | Generating code using aws/amazon.nova-pro-v1:0
2026-03-15 02:01:41 | INFO     | src.rag.generator:generate:308 | ✅ Code generation completed


✅ Success
Code preview:
  from braket.circuits import Circuit
  
  # Initialize a 3-qubit circuit
  circuit = Circuit()
  
  # Apply a Hadamard gate to the first qubit to create superposition
  circuit.h(0)
  
  # Apply a controlled-NOT gate from the first qubit to the second qubit
  circuit.cnot(0, 1)
  ...
